<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/LowValidation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

In [1]:
import transformer_lens

In [1]:
!pip install transformer_lens

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.0/192.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
  

In [ ]:
# What are we trying to create for a dataset:
# A0a, B1b, C0c, A1d,
# Given that these represent Subject - Relation - Attribute, so A0a could be: A(ris) 0(lives) a(arizona)

In [ ]:
E = 100 # num entities
A = 100 # num attributes
T = 5 # num types/relations
SEP = 205 # as seperator between relations
Q = 206 # question token
PAD = 207
D_VOCAB = 208

N_WORLDS = 20000 # (also dataset size)
MIN_FACTS, MAX_FACTS = 4, 8
SEED = 0

ENTITIES   = np.arange(0, E) # 0..99
ATTRS      = np.arange(100, 100 + A) # 100..199
TYPES      = np.arange(200, 200 + T) # 200..204

rng = np.random.default_rng(SEED)

def attr_for(world_id: int, e_tok: int, t_tok: int) -> int:
    e = e_tok
    t = t_tok - 200
    idx = (e * 31 + t * 17 + world_id * 53) % A
    return 100 + idx

def produce_example(world_id: int, num_relations: int):

    # all possible entity, type combinations
    all_relations = [(int(e), int(t)) for e in ENTITIES for t in TYPES]

    # choose *num_relations* many
    chosen = rng.choice(len(all_relations), size=num_relations, replace=False)
    relations = [all_relations[i] for i in chosen]

    # Facts: (E, T, A) with world-specific A
    facts = [(e, t, attr_for(world_id, e, t)) for (e, t) in relations]

    # Pick one fact to query for
    q_idx = rng.integers(0, num_relations)
    Eq, Tq, Aq = facts[q_idx]

    # Serialize: E T A SEP ... SEP Eq Tq Q
    seq = []
    for (e, t, a) in facts:
        seq.extend([e, t, a, SEP])
    seq.extend([Eq, Tq, Q])

    label = Aq
    return seq, label

rows = []
for w in range(N_WORLDS):
    num_relations = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label = produce_example(w, num_relations)
    rows.append({"world_id": w, "tokens": seq, "label": label})

df = pd.DataFrame(rows)

In [ ]:
df

,world_id,tokens,label
0,0,"[62, 204, 190, 205, 50, 202, 184, 205, 30, 202...",151
1,1,"[81, 201, 181, 205, 92, 203, 156, 205, 55, 202...",167
2,2,"[29, 204, 173, 205, 8, 204, 122, 205, 48, 200,...",154
3,3,"[25, 202, 168, 205, 99, 202, 162, 205, 60, 204...",137
4,4,"[38, 202, 124, 205, 57, 202, 113, 205, 71, 204...",124
...,...,...,...
19995,19995,"[64, 200, 119, 205, 52, 201, 164, 205, 59, 203...",164
19996,19996,"[18, 204, 114, 205, 19, 204, 145, 205, 68, 204...",131
19997,19997,"[2, 200, 103, 205, 63, 204, 162, 205, 79, 202,...",124
19998,19998,"[26, 203, 151, 205, 68, 200, 102, 205, 16, 200...",166


In [ ]:
### Model

In [ ]:
from transformer_lens import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=2,
    d_model=256,
    d_head=256,
    d_mlp=1024,
    n_ctx=64,
    d_vocab=208,
    act_fn="gelu",
    attn_only=False,
    normalization_type="LN",
)
model = HookedTransformer(cfg)

In [ ]:
import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

In [ ]:
IGNORE_INDEX = -100

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [ ]:
def train_collate_fn(batch, rng=np.random.default_rng()):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = shuffle_facts(seq.tolist() if torch.is_tensor(seq) else seq, rng)
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def val_collate_fn(batch):
    max_len = max(len(seq) for seq,_ in batch)
    B = len(batch)
    toks   = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = seq.tolist() if torch.is_tensor(seq) else seq
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target


In [ ]:
def shuffle_facts(seq, rng=np.random.default_rng()):
    seps = [i for i,t in enumerate(seq) if t == SEP]
    context = seq[:seps[-1]+1]
    query_part = seq[seps[-1]+1:]
    Eq, Tq = query_part[0], query_part[1]
    facts = [context[i:i+4] for i in range(0, len(context), 4)]
    rng.shuffle(facts)

    return [x for f in facts for x in f] + query_part

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = EntityBindingDataset(train_df)
val_dataset = EntityBindingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=train_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_collate_fn)

In [ ]:
len(train_dataset)

14000

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.98), weight_decay=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
writer = SummaryWriter("runs/entity_binding_hooked_transformer")

Moving model to device:  cuda


In [ ]:
def compute_accuracy(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int = IGNORE_INDEX):
    """
    Accuracy over positions where targets != ignore_index.
    Returns correct_count and total_count
    """
    with torch.no_grad():
        mask = targets.ne(ignore_index)
        total = mask.sum().item()
        preds = logits.argmax(dim=-1)
        correct = preds.masked_select(mask).eq(targets.masked_select(mask)).sum().item()
        return correct, total

In [ ]:
num_epochs = 20
global_step = 0

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    total_train_correct = 0
    total_train_count = 0
    for input_tokens, targets in train_loader:
        input_tokens, targets = input_tokens.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(input_tokens)  # shape: [B, T, d_vocab]
        loss = criterion(logits.view(-1, logits.size(-1)),
                          targets.view(-1))
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        c, n = compute_accuracy(logits, targets)
        total_train_correct += c
        total_train_count += n
        step_acc = (c / n) if n else 0.0
        writer.add_scalar("Loss/train", loss.item(), global_step)
        writer.add_scalar("Acc/train_step", step_acc, global_step)
        global_step += 1

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = (total_train_correct / total_train_count) if total_train_count else 0.0
    print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f}")

    model.eval()
    total_val_loss = 0
    total_val_correct = 0
    total_val_count = 0
    with torch.no_grad():
        for input_tokens, targets in val_loader:
            input_tokens, targets = input_tokens.to(device), targets.to(device)
            logits = model(input_tokens)
            loss = criterion(logits.view(-1, logits.size(-1)),
                          targets.view(-1))
            total_val_loss += loss.item()
            c, n = compute_accuracy(logits, targets)
            total_val_correct += c
            total_val_count += n

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0
    print(f"[Epoch {epoch+1}] Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}")
    writer.add_scalar("Loss/val", avg_val_loss, global_step)
    writer.add_scalar("Acc/val", val_acc, global_step)

writer.close()
print("Training complete.")

[Epoch 1] Train Loss: 3.6104 | Train Acc: 0.153
[Epoch 1] Val Loss: 2.9647 | Val Acc: 0.188
[Epoch 2] Train Loss: 2.5995 | Train Acc: 0.238
[Epoch 2] Val Loss: 2.7924 | Val Acc: 0.187
[Epoch 3] Train Loss: 2.3129 | Train Acc: 0.273
[Epoch 3] Val Loss: 2.6795 | Val Acc: 0.190
[Epoch 4] Train Loss: 2.0610 | Train Acc: 0.323
[Epoch 4] Val Loss: 2.6717 | Val Acc: 0.193
[Epoch 5] Train Loss: 1.8525 | Train Acc: 0.377
[Epoch 5] Val Loss: 2.6725 | Val Acc: 0.195
[Epoch 6] Train Loss: 1.6013 | Train Acc: 0.453
[Epoch 6] Val Loss: 2.7620 | Val Acc: 0.181
[Epoch 7] Train Loss: 1.3544 | Train Acc: 0.538
[Epoch 7] Val Loss: 2.8530 | Val Acc: 0.182
[Epoch 8] Train Loss: 1.0929 | Train Acc: 0.627
[Epoch 8] Val Loss: 2.9745 | Val Acc: 0.172
[Epoch 9] Train Loss: 0.8370 | Train Acc: 0.728
[Epoch 9] Val Loss: 3.1660 | Val Acc: 0.177
[Epoch 10] Train Loss: 0.6265 | Train Acc: 0.804
[Epoch 10] Val Loss: 3.2277 | Val Acc: 0.198
[Epoch 11] Train Loss: 0.4612 | Train Acc: 0.861
[Epoch 11] Val Loss: 3.4588 |

In [2]:
"""Fixed version to prevent overfitting

Key changes:
1. Removed world_id dependency in attr_for to prevent memorization
2. Added regularization techniques (weight decay)
3. Increased dataset diversity with more entities/attributes
4. Added better validation strategy
5. Improved model architecture for generalization
"""

import numpy as np
import pandas as pd

# !pip install transformer_lens

# Dataset parameters - increased for better generalization
E = 200  # num entities (increased)
A = 200  # num attributes (increased)
T = 10   # num types/relations (increased)
SEP = 405 # separator token (updated for larger vocab)
Q = 406   # question token
PAD = 407
D_VOCAB = 408 # This will be used as the base for calculating the max token ID

N_WORLDS = 30000  # increased dataset size
MIN_FACTS, MAX_FACTS = 3, 12  # wider range for more diversity
SEED = 42

ENTITIES = np.arange(0, E)
ATTRS = np.arange(E, E + A)  # 200..399
TYPES = np.arange(E + A, E + A + T)  # 400..409

rng = np.random.default_rng(SEED)

def attr_for(e_tok: int, t_tok: int) -> int:
    """
    Deterministic mapping from (entity, type) -> attribute
    Removed world_id dependency to prevent memorization
    """
    e = e_tok
    t = t_tok - (E + A)  # normalize type to 0-based
    # Use a simple but consistent mapping
    idx = (e * 37 + t * 23) % A
    return E + idx

def produce_example(world_id: int, num_relations: int):
    """Generate a single training example"""

    # All possible entity, type combinations
    all_relations = [(int(e), int(t)) for e in ENTITIES for t in TYPES]

    # Choose *num_relations* many without replacement
    chosen = rng.choice(len(all_relations), size=num_relations, replace=False)
    relations = [all_relations[i] for i in chosen]

    # Facts: (E, T, A) with consistent mapping
    facts = [(e, t, attr_for(e, t)) for (e, t) in relations]

    # Pick one fact to query for
    q_idx = rng.integers(0, num_relations)
    Eq, Tq, Aq = facts[q_idx]

    # Serialize: E T A SEP ... SEP Eq Tq Q
    seq = []
    for (e, t, a) in facts:
        seq.extend([e, t, a, SEP])
    seq.extend([Eq, Tq, Q])

    label = Aq
    return seq, label

# Generate dataset
rows = []
for w in range(N_WORLDS):
    num_relations = int(rng.integers(MIN_FACTS, MAX_FACTS + 1))
    seq, label = produce_example(w, num_relations)
    rows.append({"world_id": w, "tokens": seq, "label": label})

df = pd.DataFrame(rows)

### Improved Model with regularization

from transformer_lens import HookedTransformer, HookedTransformerConfig

cfg = HookedTransformerConfig(
    n_layers=2,
    n_heads=1,   # more attention heads
    d_model=256,
    d_head=64,   # smaller head dimension for better generalization
    d_mlp=512,   # smaller MLP to prevent overfitting
    n_ctx=128,   # larger context for longer sequences
    d_vocab=max(TYPES) + 1, # Set vocab size to max token ID + 1
    act_fn="gelu",
    attn_only=False,
    normalization_type="LN",
    use_attn_scale=True,
)
model = HookedTransformer(cfg)

import os
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split

IGNORE_INDEX = -100

class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

def train_collate_fn(batch, rng=np.random.default_rng()):
    max_len = max(len(seq) for seq, _ in batch)
    B = len(batch)
    toks = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = shuffle_facts(seq.tolist() if torch.is_tensor(seq) else seq, rng)
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def val_collate_fn(batch):
    max_len = max(len(seq) for seq, _ in batch)
    B = len(batch)
    toks = torch.full((B, max_len), PAD, dtype=torch.long)
    target = torch.full((B, max_len), IGNORE_INDEX, dtype=torch.long)

    for i, (seq, label) in enumerate(batch):
        s = seq.tolist() if torch.is_tensor(seq) else seq
        L = len(s)
        toks[i, :L] = torch.tensor(s, dtype=torch.long)
        target[i, L-1] = label
    return toks, target

def shuffle_facts(seq, rng=np.random.default_rng()):
    """Shuffle the order of facts to prevent position-based memorization"""
    seps = [i for i, t in enumerate(seq) if t == SEP]
    if not seps:
        return seq

    context = seq[:seps[-1]+1]
    query_part = seq[seps[-1]+1:]

    # Extract facts (each is 4 tokens: E, T, A, SEP)
    facts = [context[i:i+4] for i in range(0, len(context), 4) if i+3 < len(context)]
    rng.shuffle(facts)

    return [x for f in facts for x in f] + query_part

# Better train/val/test split with different world_ids
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42, stratify=None)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Test size: {len(test_df)}")

train_dataset = EntityBindingDataset(train_df)
val_dataset = EntityBindingDataset(val_df)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, collate_fn=train_collate_fn)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, collate_fn=val_collate_fn)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Improved optimizer settings
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,  # lower learning rate
    betas=(0.9, 0.95),
    weight_decay=0.1,  # higher weight decay
    eps=1e-8
)

# Add learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)
writer = SummaryWriter("runs/entity_binding_improved")

def compute_accuracy(logits: torch.Tensor, targets: torch.Tensor, ignore_index: int = IGNORE_INDEX):
    """Accuracy over positions where targets != ignore_index"""
    with torch.no_grad():
        mask = targets.ne(ignore_index)
        total = mask.sum().item()
        if total == 0:
            return 0, 0
        preds = logits.argmax(dim=-1)
        correct = preds.masked_select(mask).eq(targets.masked_select(mask)).sum().item()
        return correct, total

# Training loop with early stopping
num_epochs = 50
global_step = 0
best_val_acc = 0
patience = 10
patience_counter = 0

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0
    total_train_correct = 0
    total_train_count = 0

    for batch_idx, (input_tokens, targets) in enumerate(train_loader):
        input_tokens, targets = input_tokens.to(device), targets.to(device)

        optimizer.zero_grad()
        logits = model(input_tokens)  # shape: [B, T, d_vocab]
        loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))

        # Add label smoothing to prevent overconfidence
        # loss = loss * 0.9 + 0.1 * (-logits.log_softmax(dim=-1).mean())

        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_train_loss += loss.item()
        c, n = compute_accuracy(logits, targets)
        total_train_correct += c
        total_train_count += n

        if batch_idx % 100 == 0:
            step_acc = (c / n) if n else 0.0
            writer.add_scalar("Loss/train_step", loss.item(), global_step)
            writer.add_scalar("Acc/train_step", step_acc, global_step)

        global_step += 1

    # Update learning rate
    scheduler.step()

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = (total_train_correct / total_train_count) if total_train_count else 0.0

    # Validation
    model.eval()
    total_val_loss = 0
    total_val_correct = 0
    total_val_count = 0

    with torch.no_grad():
        for input_tokens, targets in val_loader:
            input_tokens, targets = input_tokens.to(device), targets.to(device)
            logits = model(input_tokens)
            loss = criterion(logits.view(-1, logits.size(-1)), targets.view(-1))
            total_val_loss += loss.item()
            c, n = compute_accuracy(logits, targets)
            total_val_correct += c
            total_val_count += n

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = (total_val_correct / total_val_count) if total_val_count else 0.0

    print(f"[Epoch {epoch+1:2d}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    writer.add_scalar("Loss/train", avg_train_loss, epoch)
    writer.add_scalar("Loss/val", avg_val_loss, epoch)
    writer.add_scalar("Acc/train", train_acc, epoch)
    writer.add_scalar("Acc/val", val_acc, epoch)
    writer.add_scalar("LR", scheduler.get_last_lr()[0], epoch)

    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), "best_model.pt")
    else:
        patience_counter += 1

    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

writer.close()
print(f"Training complete. Best validation accuracy: {best_val_acc:.3f}")

# Load best model for final evaluation
model.load_state_dict(torch.load("best_model.pt"))

# Test the model's generalization
def test_model_on_new_examples(model, num_test_examples=1000):
    """Test model on completely new examples not seen during training"""
    test_rng = np.random.default_rng(12345)  # different seed
    test_rows = []

    for w in range(num_test_examples):
        num_relations = int(test_rng.integers(MIN_FACTS, MAX_FACTS + 1))
        seq, label = produce_example(w + N_WORLDS, num_relations)  # new world_ids
        test_rows.append({"world_id": w + N_WORLDS, "tokens": seq, "label": label})

    test_df_new = pd.DataFrame(test_rows)
    test_dataset_new = EntityBindingDataset(test_df_new)
    test_loader_new = DataLoader(test_dataset_new, batch_size=64, shuffle=False, collate_fn=val_collate_fn)

    model.eval()
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for input_tokens, targets in test_loader_new:
            input_tokens, targets = input_tokens.to(device), targets.to(device)
            logits = model(input_tokens)
            c, n = compute_accuracy(logits, targets)
            total_correct += c
            total_count += n

    test_acc = (total_correct / total_count) if total_count else 0.0
    print(f"Test accuracy on new examples: {test_acc:.3f}")
    return test_acc

# Test generalization
test_acc = test_model_on_new_examples(model)

Train size: 18000, Val size: 6000, Test size: 6000
Moving model to device:  cuda
[Epoch  1] Train Loss: 5.3555 | Train Acc: 0.026 | Val Loss: 4.8770 | Val Acc: 0.088 | LR: 9.99e-05
[Epoch  2] Train Loss: 4.3679 | Train Acc: 0.149 | Val Loss: 4.1363 | Val Acc: 0.163 | LR: 9.96e-05
[Epoch  3] Train Loss: 3.6291 | Train Acc: 0.231 | Val Loss: 3.6371 | Val Acc: 0.209 | LR: 9.91e-05
[Epoch  4] Train Loss: 3.0135 | Train Acc: 0.338 | Val Loss: 3.0692 | Val Acc: 0.313 | LR: 9.84e-05
[Epoch  5] Train Loss: 2.3385 | Train Acc: 0.503 | Val Loss: 2.3950 | Val Acc: 0.451 | LR: 9.76e-05
[Epoch  6] Train Loss: 1.7414 | Train Acc: 0.641 | Val Loss: 1.8998 | Val Acc: 0.545 | LR: 9.65e-05
[Epoch  7] Train Loss: 1.3437 | Train Acc: 0.728 | Val Loss: 1.5765 | Val Acc: 0.608 | LR: 9.53e-05
[Epoch  8] Train Loss: 1.0740 | Train Acc: 0.776 | Val Loss: 1.3838 | Val Acc: 0.646 | LR: 9.39e-05
[Epoch  9] Train Loss: 0.8777 | Train Acc: 0.817 | Val Loss: 1.2453 | Val Acc: 0.661 | LR: 9.23e-05
[Epoch 10] Train Lo